# Percobaan 5 - Simple CatBoost Baseline

Baseline sederhana untuk kompetisi Gammafest football score prediction.

Fokus notebook ini:
- preprocessing missing value yang aman,
- fitur shared antara train dan test,
- validasi temporal,
- dua model CatBoostRegressor untuk `team_goals` dan `opp_goals`,
- evaluasi AW-MAE,
- post-processing integer sederhana,
- generate submission.

Catatan: dataset dibiarkan di folder `dataset/` karena notebook lama juga memakai path tersebut.

In [4]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

## 1. Path dan konfigurasi

Default model dibuat stabil dulu dengan CPU. Kalau kernel `py_gpu_ready` punya GPU CatBoost yang siap, ganti `TASK_TYPE` menjadi `"GPU"`.

In [5]:
BASE_PATH = Path.home() / 'Downloads' / 'Gammafest'
DATA_PATH = BASE_PATH / 'dataset'
OUTPUT_DIR = BASE_PATH / 'experiments' / 'percobaan 5 - simple baseline'

TRAIN_PATH = DATA_PATH / 'train.csv'
TEST_PATH = DATA_PATH / 'test.csv'
SAMPLE_PATH = DATA_PATH / 'sample submission.csv'
SUBMISSION_PATH = OUTPUT_DIR / 'submission_simple_catboost.csv'

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = 'CPU'  # ganti ke 'GPU' kalau CatBoost GPU tersedia dan ingin dicoba
MAX_SCORE = 6

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data path:', DATA_PATH)
print('Output path:', OUTPUT_DIR)

Data path: C:\Users\Gusti Jogish\Downloads\Gammafest\dataset
Output path: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline


## 2. Load data

In [6]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print('train:', train_raw.shape)
print('test :', test_raw.shape)
print('sample:', sample.shape)

display(train_raw.head(2))
display(test_raw.head(2))
display(sample.head(2))

train: (78772, 47)
test : (42422, 20)
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169


,Id,team_goals,opp_goals
0,M034984_Seychelles,0,0
1,M034984_Mauritius,0,0


## 3. Fitur baseline

Baseline ini hanya memakai fitur yang tersedia di train dan test agar tidak perlu rekonstruksi fitur historis dulu.

In [7]:
CAT_COLS = [
    'gender',
    'team',
    'opponent',
    'tournament',
    'venue_country',
    'confederation_team',
    'confederation_opp',
]

NUM_COLS = [
    'is_home',
    'neutral',
    'population_team',
    'population_opp',
    'gdp_per_capita_team',
    'gdp_per_capita_opp',
    'altitude_venue',
    'distance_travel_team',
    'distance_travel_opp',
    'temperature_venue',
]

DATE_FEATURES = ['year', 'month', 'dayofweek']
TARGET_COLS = ['team_goals', 'opp_goals']

missing_train = [c for c in CAT_COLS + NUM_COLS + ['date'] if c not in train_raw.columns]
missing_test = [c for c in CAT_COLS + NUM_COLS + ['date'] if c not in test_raw.columns]
print('Missing in train:', missing_train)
print('Missing in test :', missing_test)

Missing in train: []
Missing in test : []


## 4. Preprocessing function

Strategi missing value:
- nilai sentinel `-9999` di numeric dianggap missing,
- buat missing flag untuk numeric,
- categorical missing diisi `Unknown`,
- numeric missing diisi median dari train saja.

In [8]:
def add_date_features(df):
    df = df.copy()
    dt = pd.to_datetime(df['date'], errors='coerce')
    df['year'] = dt.dt.year
    df['month'] = dt.dt.month
    df['dayofweek'] = dt.dt.dayofweek
    return df


def prepare_features(train_df, test_df, cat_cols, num_cols):
    train = add_date_features(train_df)
    test = add_date_features(test_df)

    numeric_base = num_cols + DATE_FEATURES

    # Sentinel value cleanup. In this dataset, -9999 appears as missing-like altitude.
    for df in [train, test]:
        for col in numeric_base:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                df.loc[df[col] == -9999, col] = np.nan

    # Missing flags are created before imputation.
    missing_flag_cols = []
    for col in numeric_base:
        flag_col = f'{col}_missing'
        train[flag_col] = train[col].isna().astype(int)
        test[flag_col] = test[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    # Impute numeric columns using train median only.
    medians = train[numeric_base].median(numeric_only=True)
    for col in numeric_base:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)

    # CatBoost categorical columns should be strings.
    for col in cat_cols:
        train[col] = train[col].fillna('Unknown').astype(str)
        test[col] = test[col].fillna('Unknown').astype(str)

    feature_cols = cat_cols + numeric_base + missing_flag_cols
    return train, test, feature_cols, missing_flag_cols, medians

train_prep, test_prep, FEATURE_COLS, MISSING_FLAG_COLS, TRAIN_MEDIANS = prepare_features(
    train_raw, test_raw, CAT_COLS, NUM_COLS
)

print('Total features:', len(FEATURE_COLS))
print('Categorical:', CAT_COLS)
print('Missing flags:', len(MISSING_FLAG_COLS))
print(FEATURE_COLS)

Total features: 33
Categorical: ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']
Missing flags: 13
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'dayofweek', 'is_home_missing', 'neutral_missing', 'population_team_missing', 'population_opp_missing', 'gdp_per_capita_team_missing', 'gdp_per_capita_opp_missing', 'altitude_venue_missing', 'distance_travel_team_missing', 'distance_travel_opp_missing', 'temperature_venue_missing', 'year_missing', 'month_missing', 'dayofweek_missing']


## 5. Quick missing-value check

In [9]:
missing_report = pd.DataFrame({
    'train_missing_after': train_prep[FEATURE_COLS].isna().sum(),
    'test_missing_after': test_prep[FEATURE_COLS].isna().sum(),
})

display(missing_report[missing_report.sum(axis=1) > 0])

flag_activity = test_prep[MISSING_FLAG_COLS].sum().sort_values(ascending=False)
display(flag_activity.head(20).to_frame('test_missing_count'))

,train_missing_after,test_missing_after


,test_missing_count
distance_travel_opp_missing,16975
distance_travel_team_missing,16975
gdp_per_capita_opp_missing,14664
gdp_per_capita_team_missing,14664
altitude_venue_missing,11202
temperature_venue_missing,5594
population_opp_missing,3372
population_team_missing,3372
is_home_missing,0
neutral_missing,0


## 6. Temporal validation split

Data diurutkan berdasarkan tanggal. Bagian terbaru dipakai sebagai validation agar lebih mirip kondisi prediksi masa depan.

In [10]:
train_prep['date_dt'] = pd.to_datetime(train_prep['date'], errors='coerce')
train_sorted = train_prep.sort_values('date_dt').reset_index(drop=True)

split_idx = int(len(train_sorted) * (1 - VALID_FRAC))
tr_df = train_sorted.iloc[:split_idx].copy()
val_df = train_sorted.iloc[split_idx:].copy()

print('Train fold:', tr_df.shape, tr_df['date_dt'].min(), '->', tr_df['date_dt'].max())
print('Valid fold:', val_df.shape, val_df['date_dt'].min(), '->', val_df['date_dt'].max())

X_tr = tr_df[FEATURE_COLS]
X_val = val_df[FEATURE_COLS]
y_tr_team = tr_df['team_goals']
y_tr_opp = tr_df['opp_goals']
y_val_team = val_df['team_goals']
y_val_opp = val_df['opp_goals']

cat_feature_indices = [FEATURE_COLS.index(c) for c in CAT_COLS]
cat_feature_indices

Train fold: (63017, 64) 1872-11-30 00:00:00 -> 2005-02-01 00:00:00
Valid fold: (15755, 64) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00


[0, 1, 2, 3, 4, 5, 6]

## 7. AW-MAE metric

Implementasi mengikuti deskripsi lomba:
- MAE skor,
- exact score penalty,
- outcome penalty,
- goal difference penalty,
- outcome multiplier,
- non-linear scaling pangkat 1.5,
- tournament weight.

In [11]:
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 2.00,
    'AFC Championship': 1.80,
    'AFC Asian Cup': 1.80,
    'UEFA Euro': 1.80,
    'Copa America': 1.80,
    'Copa Am?rica': 1.80,
    'Africa Cup of Nations': 1.80,
    'African Cup of Nations': 1.80,
    'Gold Cup': 1.75,
    'CONCACAF Gold Cup': 1.75,
    'FIFA World Cup qualification': 1.50,
    'UEFA Euro qualification': 1.40,
    'AFC Asian Cup qualification': 1.40,
    'Friendly': 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    team = np.clip(team, 0, max_score)
    opp = np.clip(opp, 0, max_score)
    return team, opp


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true['team_goals'].to_numpy()
    true_opp = df_true['opp_goals'].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2

    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    raw_loss = mae + penalty
    loss = (raw_loss * multiplier) ** 1.5

    weights = df_true['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()
    score = np.sum(loss * weights) / np.sum(weights)

    diagnostics = {
        'AW-MAE': score,
        'MAE_raw': mae.mean(),
        'exact_acc': exact.mean(),
        'outcome_acc': outcome.mean(),
        'goal_diff_acc': gd.mean(),
    }
    return score, diagnostics

## 8. Train CatBoost baseline

In [12]:
cat_params = dict(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=1200,
    learning_rate=0.045,
    depth=7,
    l2_leaf_reg=6,
    random_seed=RANDOM_STATE,
    task_type=TASK_TYPE,
    verbose=100,
    allow_writing_files=False,
)

model_team = CatBoostRegressor(**cat_params)
model_opp = CatBoostRegressor(**cat_params)

print('Training team_goals model...')
model_team.fit(
    X_tr, y_tr_team,
    cat_features=cat_feature_indices,
    eval_set=(X_val, y_val_team),
    use_best_model=True,
    early_stopping_rounds=120,
)

print('Training opp_goals model...')
model_opp.fit(
    X_tr, y_tr_opp,
    cat_features=cat_feature_indices,
    eval_set=(X_val, y_val_opp),
    use_best_model=True,
    early_stopping_rounds=120,
)

Training team_goals model...
0:	learn: 1.1707540	test: 1.1294111	best: 1.1294111 (0)	total: 138ms	remaining: 2m 45s
100:	learn: 1.0680912	test: 1.0528929	best: 1.0528929 (100)	total: 5.66s	remaining: 1m 1s
200:	learn: 1.0489933	test: 1.0444516	best: 1.0444516 (200)	total: 10.6s	remaining: 52.9s
300:	learn: 1.0326976	test: 1.0378234	best: 1.0378234 (300)	total: 15.6s	remaining: 46.6s
400:	learn: 1.0201265	test: 1.0342917	best: 1.0341760 (395)	total: 20.4s	remaining: 40.7s
500:	learn: 1.0100368	test: 1.0337291	best: 1.0335064 (483)	total: 25.2s	remaining: 35.1s
600:	learn: 1.0009336	test: 1.0324205	best: 1.0323849 (598)	total: 30.6s	remaining: 30.5s
700:	learn: 0.9936176	test: 1.0322423	best: 1.0320404 (655)	total: 35.8s	remaining: 25.5s
Stopped by overfitting detector  (120 iterations wait)

bestTest = 1.032040431
bestIteration = 655

Shrink model to first 656 iterations.
Training opp_goals model...
0:	learn: 1.1710106	test: 1.1298761	best: 1.1298761 (0)	total: 55.3ms	remaining: 1m 6s
1

## 9. Validasi baseline

Kita bandingkan prediksi mentah dan hasil round+clip sederhana.

In [15]:
val_pred_team_raw = np.clip(model_team.predict(X_val), 0, None)
val_pred_opp_raw = np.clip(model_opp.predict(X_val), 0, None)

val_pred_team_int, val_pred_opp_int = postprocess_round_clip(
    val_pred_team_raw, val_pred_opp_raw, max_score=MAX_SCORE
)

mae_team_raw = mean_absolute_error(y_val_team, val_pred_team_raw)
mae_opp_raw = mean_absolute_error(y_val_opp, val_pred_opp_raw)
mae_team_int = mean_absolute_error(y_val_team, val_pred_team_int)
mae_opp_int = mean_absolute_error(y_val_opp, val_pred_opp_int)

awmae_score, diag = compute_awmae(val_df, val_pred_team_int, val_pred_opp_int)

print('Raw MAE team:', mae_team_raw)
print('Raw MAE opp :', mae_opp_raw)
print('Int MAE team:', mae_team_int)
print('Int MAE opp :', mae_opp_int)
print('AW-MAE diagnostics:')
for k, v in diag.items():
    print(f'{k}: {v:.6f}')

Raw MAE team: 1.030802206689104
Raw MAE opp : 1.0303649212337647
Int MAE team: 1.010599809584259
Int MAE opp : 1.0124404950809267
AW-MAE diagnostics:
AW-MAE: 3.163438
MAE_raw: 1.011520
exact_acc: 0.106125
outcome_acc: 0.502317
goal_diff_acc: 0.236496


## 10. Simple error inspection

In [16]:
val_view = val_df[['Id', 'date', 'team', 'opponent', 'tournament', 'team_goals', 'opp_goals']].copy()
val_view['pred_team_goals'] = val_pred_team_int
val_view['pred_opp_goals'] = val_pred_opp_int
val_view['abs_err'] = (
    (val_view['team_goals'] - val_view['pred_team_goals']).abs()
    + (val_view['opp_goals'] - val_view['pred_opp_goals']).abs()
) / 2

display(val_view.sort_values('abs_err', ascending=False).head(20))

display(
    val_view.groupby('tournament')['abs_err']
    .agg(['count', 'mean'])
    .sort_values(['count', 'mean'], ascending=[False, False])
    .head(20)
)

,Id,date,team,opponent,tournament,team_goals,opp_goals,pred_team_goals,pred_opp_goals,abs_err
66778,W002963_South Africa,2006-11-10,South Africa,Cameroon,African Championship,24,2,1,2,11.5
66780,W002963_Cameroon,2006-11-10,Cameroon,South Africa,African Championship,2,24,2,1,11.5
75014,W003840_Palestine,2010-02-22,Palestine,Kuwait,WAFF Championship,17,0,0,3,10.0
67060,M030514_Monaco,2006-11-24,Monaco,Sápmi,Viva World Cup,1,21,2,2,10.0
67065,M030514_Sápmi,2006-11-24,Sápmi,Monaco,Viva World Cup,21,1,1,1,10.0
75015,W003840_Kuwait,2010-02-22,Kuwait,Palestine,WAFF Championship,0,17,3,0,10.0
69181,W003291_British Virgin Islands,2007-10-03,British Virgin Islands,Cuba,CONCACAF Olympic Qualifying Tournament qualifi...,0,21,0,4,8.5
69182,W003291_Cuba,2007-10-03,Cuba,British Virgin Islands,CONCACAF Olympic Qualifying Tournament qualifi...,21,0,4,0,8.5
63218,M029041_Guam,2005-03-11,Guam,North Korea,EAFF Championship,0,21,0,4,8.5
63213,M029041_North Korea,2005-03-11,North Korea,Guam,EAFF Championship,21,0,4,0,8.5


,count,mean
tournament,,
Friendly,4633,0.866825
FIFA World Cup qualification,3088,0.970531
UEFA Euro qualification,1256,0.955016
African Cup of Nations qualification,444,0.822072
FIFA World Cup,384,0.925781
AFC Asian Cup qualification,346,1.115607
Island Games,328,1.396341
Algarve Cup,328,1.025915
CECAFA Cup,268,0.919776


## 11. Train final model on all train

Setelah validasi baseline masuk akal, final model dilatih memakai seluruh train.

In [17]:
X_full = train_prep[FEATURE_COLS]
y_full_team = train_prep['team_goals']
y_full_opp = train_prep['opp_goals']
X_test = test_prep[FEATURE_COLS]

final_params = cat_params.copy()
final_params['iterations'] = int(max(model_team.best_iteration_, model_opp.best_iteration_) + 100)
final_params.pop('eval_metric', None)

print('Final iterations:', final_params['iterations'])

final_team = CatBoostRegressor(**final_params)
final_opp = CatBoostRegressor(**final_params)

print('Training final team_goals model...')
final_team.fit(X_full, y_full_team, cat_features=cat_feature_indices, verbose=100)

print('Training final opp_goals model...')
final_opp.fit(X_full, y_full_opp, cat_features=cat_feature_indices, verbose=100)

Final iterations: 950
Training final team_goals model...
0:	learn: 1.1628650	total: 59.2ms	remaining: 56.2s
100:	learn: 1.0615889	total: 5.42s	remaining: 45.5s
200:	learn: 1.0427558	total: 11s	remaining: 40.8s
300:	learn: 1.0299408	total: 16.3s	remaining: 35.1s
400:	learn: 1.0170651	total: 24.9s	remaining: 34.1s
500:	learn: 1.0073472	total: 33.6s	remaining: 30.1s
600:	learn: 0.9985444	total: 42s	remaining: 24.4s
700:	learn: 0.9922277	total: 50.1s	remaining: 17.8s
800:	learn: 0.9866580	total: 59.3s	remaining: 11s
900:	learn: 0.9811088	total: 1m 8s	remaining: 3.71s
949:	learn: 0.9786355	total: 1m 12s	remaining: 0us
Training final opp_goals model...
0:	learn: 1.1629667	total: 112ms	remaining: 1m 46s
100:	learn: 1.0605422	total: 8.84s	remaining: 1m 14s
200:	learn: 1.0407913	total: 18.3s	remaining: 1m 8s
300:	learn: 1.0271513	total: 27.7s	remaining: 59.7s
400:	learn: 1.0156696	total: 37.2s	remaining: 50.9s
500:	learn: 1.0067487	total: 46.8s	remaining: 42s
600:	learn: 0.9992190	total: 56s	re

## 12. Generate submission

In [18]:
test_pred_team_raw = np.clip(final_team.predict(X_test), 0, None)
test_pred_opp_raw = np.clip(final_opp.predict(X_test), 0, None)

test_pred_team_int, test_pred_opp_int = postprocess_round_clip(
    test_pred_team_raw, test_pred_opp_raw, max_score=MAX_SCORE
)

submission = sample[['Id']].copy()
submission['team_goals'] = test_pred_team_int
submission['opp_goals'] = test_pred_opp_int

# Keep sample order and shape.
assert submission.shape == sample.shape
assert submission['Id'].equals(sample['Id'])

submission.to_csv(SUBMISSION_PATH, index=False)
print('Saved:', SUBMISSION_PATH)
display(submission.head())
display(submission[['team_goals', 'opp_goals']].describe())

Saved: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline\submission_simple_catboost.csv


,Id,team_goals,opp_goals
0,M034984_Seychelles,1,2
1,M034984_Mauritius,1,1
2,M034985_Comoros,1,1
3,M034985_Maldives,1,1
4,M034986_Réunion,1,1


,team_goals,opp_goals
count,42422.000000,42422.000000
mean,1.317995,1.323064
std,0.953854,0.963752
min,0.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,6.000000,6.000000


## 13. Next upgrade ideas

Setelah baseline ini jalan, upgrade yang paling masuk akal:
1. outcome-aware post-processing,
2. rekonstruksi historical features untuk test,
3. model classifier win/draw/loss,
4. ensemble CatBoost + XGBoost/LightGBM.